In [1]:
# 将下面的 "你的真实密钥" 替换为你个人的 API Key

api_key = "sk-7e1a80312250477bb0d63301dc555946" 

content = f"""BASE_URL=https://chat.ecnu.edu.cn/open/api/v1
MODEL=ecnu-plus
API_KEY={api_key}
"""

# 写入 .env 文件

with open(".env", "w", encoding="utf-8") as f:
    f.write(content)

print("✅ .env 文件已在当前目录创建成功！")

✅ .env 文件已在当前目录创建成功！


In [2]:
# 检查env文件是否创建   win的指令是!dir /a
!ls -a

.                  .DS_Store          .ipynb_checkpoints test.ipynb
..                 .env               chat_history.json


In [3]:
# 查看内容     win的指令是   !type .env
!cat .env

BASE_URL=https://chat.ecnu.edu.cn/open/api/v1
MODEL=ecnu-plus
API_KEY=sk-7e1a80312250477bb0d63301dc555946


In [4]:
# 安装openai依赖 读取env内容。 或者 !pip install openai python-dotenv -i https://pypi.tuna.tsinghua.edu.cn/simple
! pip install openai python-dotenv

In [5]:
# 导入依赖
import os
#读取env文件
from dotenv import load_dotenv
#openai协议
from openai import OpenAI

In [6]:
# 读取环境
load_dotenv(".env", override=True)
client = OpenAI(
api_key=os.environ["API_KEY"],
base_url=os.environ["BASE_URL"],
)

In [7]:
def ask_llm(question, system_prompt="你是一个AI助手,回答尽量简短。",temperature=0.7, max_tokens=2000):

    # 构建消息格式：第一条是系统提示词，第二条是用户提问
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    
    # 发送请求
    response = client.chat.completions.create(
        model=os.environ["MODEL"],
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        stream=False,
    )
    
    # 提取回答内容
    answer = response.choices[0].message.content
    
    # 打印结果（格式化输出）
    print("提问：", question)
    print("回答：", answer)
    
    # 返回回答，方便后续可能需要对回答做其他处理
    return answer

In [8]:
# 第一次提问
ask_llm("我正在上大模型原理的课程")

提问： 我正在上大模型原理的课程
回答： 欢迎！大模型原理是一门前沿且充满挑战的课程。

你目前具体想了解哪个模块？例如：
*   **基础架构**：Transformer、Attention机制
*   **训练过程**：预训练、微调（SFT/RLHF）、优化器
*   **推理优化**：KV Cache、量化、蒸馏
*   **前沿技术**：MoE、多模态、长上下文处理

请提出具体问题，我会为你简明解答。


'欢迎！大模型原理是一门前沿且充满挑战的课程。\n\n你目前具体想了解哪个模块？例如：\n*   **基础架构**：Transformer、Attention机制\n*   **训练过程**：预训练、微调（SFT/RLHF）、优化器\n*   **推理优化**：KV Cache、量化、蒸馏\n*   **前沿技术**：MoE、多模态、长上下文处理\n\n请提出具体问题，我会为你简明解答。'

In [9]:
# 第二次提问
ask_llm("我现在在干嘛")

提问： 我现在在干嘛
回答： 你在和我对话。


'你在和我对话。'

In [10]:
# 但是我们平时使用到的GPT，其实你问的下一个问题，都是和上面的问题有关联的，所以，多轮对话的设计才更有意义。为了方便，我们将其设置为 类 的形式

In [10]:
class ChatBot:
    def __init__(self, system_prompt="你是一个AI助手,回答尽量简短。"):
        # 初始化历史记录，默认包含一条 system 消息
        self.history = [
            {"role": "system", "content": system_prompt}
        ]
    
    def ask(self, question,temperature=0.7, max_tokens=20000):
        """
        提问函数：默认在原有历史记录上继续对话
        """
        # 1. 把用户的新问题加入历史
        self.history.append({"role": "user", "content": question})
        
        try:
            # 2. 发送请求，把整个历史记录传给模型
            response = client.chat.completions.create(
                model=os.environ["MODEL"],
                messages=self.history,
                temperature=temperature,
                max_tokens=max_tokens,
                stream=False,
            )
            
            # 3. 提取回答
            answer = response.choices[0].message.content
            
            # 4. 把模型的回答也加入历史（这样它下次就知道自己说过什么了）
            self.history.append({"role": "assistant", "content": answer})
            
            # 打印结果
            print(f"👤 提问：{question}")
            print(f"🤖 回答：{answer}\n")
            
            return answer
            
        except Exception as e:
            print(f"❌ 请求出错：{e}")
            # 如果出错了，把刚才加入的用户问题弹出去，保持历史记录干净
            self.history.pop()
            return None
            
    def clear_history(self):
        """
        清空对话历史：把记录重置，只保留最初设定的 system 提示词
        """
        # 保留第一条 system 消息
        if self.history and self.history[0]["role"] == "system":
            self.history = [self.history[0]]
        else:
            self.history = []
        print("🧹 对话历史已清空，可以开始新的话题了！\n")
        
    def show_history(self):

        print(self.history)
        # 计算当前有多少轮对话 (总条数 - 1条system) / 2
        turns = (len(self.history) - 1) // 2 
        print(f"📊 当前已进行 {turns} 轮对话。\n")

In [11]:
# 定义一个实例
bot = ChatBot(system_prompt="你是一个AI助手，回答尽量简短。")

In [12]:
bot.ask("我正在上大模型原理的课程。")

👤 提问：我正在上大模型原理的课程。
🤖 回答：好的，请问有什么具体问题？



'好的，请问有什么具体问题？'

In [13]:
bot.ask("我现在在干嘛")

👤 提问：我现在在干嘛
🤖 回答：你正在上大模型原理的课程。



'你正在上大模型原理的课程。'

In [14]:
# 查看当前有几轮对话
bot.show_history()

[{'role': 'system', 'content': '你是一个AI助手，回答尽量简短。'}, {'role': 'user', 'content': '我正在上大模型原理的课程。'}, {'role': 'assistant', 'content': '好的，请问有什么具体问题？'}, {'role': 'user', 'content': '我现在在干嘛'}, {'role': 'assistant', 'content': '你正在上大模型原理的课程。'}]
📊 当前已进行 2 轮对话。



In [15]:
# 清空历史（开始新话题）
bot.clear_history()

🧹 对话历史已清空，可以开始新的话题了！



In [16]:
bot.ask("我现在在干嘛")

👤 提问：我现在在干嘛
🤖 回答：你在和我聊天。



'你在和我聊天。'

In [17]:
# 课程任务：请从下列主题中，选择一个，然后和大模型进行多轮对话，得出一个统一的结论（可以是你说服大模型，也可以是你认可大模型的想法，对话轮次至少5轮以上）
# 任务提交：所有的代码，以及和大模型的对话记录


In [18]:
# 主题1：“大模型在接下来的大模型发展中，究竟是会先彻底颠覆教育行业，还是先颠覆医疗行业？”

In [19]:
# 主题2：“随着大模型越来越拟人化，我们是否应该赋予其‘虚拟人格权’？”

In [20]:
# 主题3：“面对 AI 顶会投稿量暴增及大量 LLM 生成‘抽奖’稿件，学术界应采取何种防御与应对机制？”